# 001. Data Pipeline
- This gathers single-run daily data and extracts updated data from yesterday 
- Type: Pipeline
- Run Frequency: Daily (Morning)
- Created: 1/1/2025
- Updated: 5/29/2025

### Imports

In [ ]:
import sys
sys.path.append(r"C:\Users\James\Documents\MLB\Code")

In [ ]:
from U01Imports import *
from U02Functions import *
from U03Classes import *
from U04Datasets import *

from A01PlayerResults import *
from A02MLBAPI import *
from A03FanGraphs import *
from A03Steamer import *
from A04Bullpens import *
from A05Rosters import *
from A06Weather import *
from A07Odds import *
from A08Projections import *
from A09DraftKings import *

from B01Matchups import *
from B03ContestGuides import *

### Settings

In [ ]:
start_date, end_date = yesterdaysdate, todaysdate
year = int(start_date[:4])

### Games

Refresh game dataset

In [ ]:
all_game_df = create_games(team_dict=team_dict, baseball_path=baseball_path, venue_map_df=venue_map_df, refresh_start_date=yesterdaysdate, refresh_end_date=todaysdate)

Select games

In [ ]:
game_df = all_game_df[(all_game_df['date'].astype(str) >= start_date) & (all_game_df['date'].astype(str) <= end_date)].reset_index(drop=True)

Fill in missing starters

In [ ]:
game_df[['home_probable_pitcher', 'away_probable_pitcher']] = game_df[['home_probable_pitcher', 'away_probable_pitcher']].fillna("Missing Starter")

### A01. Player Results

Date(s): Yesterday

In [ ]:
df = game_df[game_df['date'].astype(str) == yesterdaysdate].reset_index(drop=True)

In [ ]:
_ = Parallel(n_jobs=-1)(delayed(run_player_results)(df, row) for row in range(len(df)))

### A02. MLB API

Date(s): Year-to-Date

##### 1. Stats API

In [ ]:
max_retries = 5

for attempt in range(max_retries):
    try:
        plays_statsapi(f"03/18/{year}", f"11/15/{year}").to_csv(os.path.join(baseball_path, "A02. MLB API", "1. Stats API", f"Stats API {year}.csv"), index=False, encoding='iso-8859-1')
        break

    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}")
        if attempt == max_retries - 1:
            raise

        time.sleep(60)

##### 2. Statcast

In [ ]:
max_retries = 5

for attempt in range(max_retries):
    try:
        plays_statcast(f"{year}-03-18", yesterdaysdate_dash).to_csv(os.path.join(baseball_path, "A02. MLB API", "2. Statcast", f"Statcast {year}.csv"), index=False, encoding='iso-8859-1')
        break

    except Exception as e:
        print(f"Attempt {attempt+1} failed: {e}")
        if attempt == max_retries - 1:
            raise
    time.sleep(30)

### A03. FanGraphs

Date(s): Daily

##### Steamer

In [ ]:
fangraphs_api("https://www.fangraphs.com/api/projections?type=steamerr&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "Steamer", "Hitters", "2026", f"Steamer Hitters {todaysdate}.csv"), index=False)
fangraphs_api("https://www.fangraphs.com/api/projections?type=steamerr&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "Steamer", "Pitchers", "2026", f"Steamer Pitchers {todaysdate}.csv"), index=False)

##### ZiPS

In [ ]:
fangraphs_api("https://www.fangraphs.com/api/projections?type=rzips&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "ZiPS", "Hitters", "2026", f"ZiPS Hitters {todaysdate}.csv"), index=False)
fangraphs_api("https://www.fangraphs.com/api/projections?type=rzips&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "ZiPS", "Pitchers", "2026", f"ZiPS Pitchers {todaysdate}.csv"), index=False)

##### THE BAT X

In [ ]:
fangraphs_api("https://www.fangraphs.com/api/projections?type=rthebatx&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "THE BAT X", "Hitters", "2026", f"THE BAT X Hitters {todaysdate}.csv"), index=False)
fangraphs_api("https://www.fangraphs.com/api/projections?type=rthebatx&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "THE BAT X", "Pitchers", "2026", f"THE BAT X Pitchers {todaysdate}.csv"), index=False)

##### ATC

In [ ]:
fangraphs_api("https://www.fangraphs.com/api/projections?type=ratcdc&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "ATC", "Hitters", "2026", f"ATC Hitters {todaysdate}.csv"), index=False)
fangraphs_api("https://www.fangraphs.com/api/projections?type=ratcdc&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "ATC", "Pitchers", "2026", f"ATC Pitchers {todaysdate}.csv"), index=False)

##### OOPSY

In [ ]:
fangraphs_api("https://www.fangraphs.com/api/projections?type=roopsydc&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "OOPSY", "Hitters", "2026", f"OOPSY Hitters {todaysdate}.csv"), index=False)
fangraphs_api("https://www.fangraphs.com/api/projections?type=roopsydc&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "OOPSY", "Pitchers", "2026", f"OOPSY Pitchers {todaysdate}.csv"), index=False)

### A03. Steamer

In [ ]:
def download_steamer():
    import requests
    from bs4 import BeautifulSoup

    session = requests.Session()

    headers = {
        'Host': 'www.steamerprojections.com',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    }

    base_url = 'http://143.95.246.118'

    # Get login page and extract hidden fields
    login_page = session.get(f'{base_url}/index.php/login', headers=headers)
    soup = BeautifulSoup(login_page.text, 'html.parser')
    hidden = soup.find_all('input', {'type': 'hidden'})
    csrf_field = {h['name']: h['value'] for h in hidden}

    # Build login payload
    login_data = {
        'username': os.getenv("STEAMER_USERNAME"),
        'password': os.getenv("STEAMER_PASSWORD"),
        'remember': 'yes',
        **csrf_field
    }

    # Login
    session.post(
        f'{base_url}/index.php/login?task=user.login',
        data=login_data,
        headers={
            **headers,
            'Referer': 'http://www.steamerprojections.com/index.php/login',
            'Content-Type': 'application/x-www-form-urlencoded',
        }
    )

    # Get projection links
    projections_page = session.get(f'{base_url}/index.php/projections/2026-projections', headers=headers)
    soup = BeautifulSoup(projections_page.text, 'html.parser')
    links = [a.get('href') for a in soup.find_all('a', href=True)]

    pitcher_links = [l for l in links if 'item_id=1168' in l]
    hitter_links = [l for l in links if 'item_id=1167' in l]

    # Download files
    for name, link in [('pitchers', pitcher_links[0]), ('hitters', hitter_links[0])]:
        url = base_url + link
        r = session.get(url, headers=headers)
        filepath = os.path.join(download_path, f'steamer_{name}.csv')
        with open(filepath, 'wb') as f:
            f.write(r.content)
        print(f"Downloaded {name}: {len(r.content)} bytes -> {filepath}")

In [ ]:
def move_steamer():
    # Find latest pitcher current download
    matching_files = glob.glob(os.path.join(download_path, "steamer_pitchers.csv"))

    # Move + rename
    most_recent_file = max(matching_files, key=os.path.getmtime)
    pitcher_destination = os.path.join(
        baseball_path,
        "A03. Steamer",
        "Pitchers",
        "2026",
        f"steamer_pitchers_{todaysdate}.csv"
    )
    shutil.move(most_recent_file, pitcher_destination)

    # Find latest hitter current download
    matching_files = glob.glob(os.path.join(download_path, "steamer_hitters.csv"))

    # Move + rename
    most_recent_file = max(matching_files, key=os.path.getmtime)
    hitter_destination = os.path.join(
        baseball_path,
        "A03. Steamer",
        "Hitters",
        "2026",
        f"steamer_hitters_{todaysdate}.csv"
    )
    shutil.move(most_recent_file, hitter_destination)

In [ ]:
download_steamer()

In [ ]:
move_steamer()

##### Hitters

In [ ]:
files = [os.path.join(baseball_path, "A03. Steamer", "Hitters", f"steamer_hitters_weekly_log_{y}.csv") for y in range(2014, year+1)] + glob.glob(os.path.join(baseball_path, "A03. Steamer", "Hitters", "2026", "*.csv"))
steamer_hitter_df = pd.concat([pd.read_csv(f, dtype=str).assign(gamedate=f"{os.path.basename(f)[-12:-8]}-{os.path.basename(f)[-8:-6]}-{os.path.basename(f)[-6:-4]}") if os.path.dirname(f).endswith("2026") else pd.read_csv(f, dtype=str) for f in files], ignore_index=True)
steamer_hitter_df.to_csv(os.path.join(baseball_path, "A03. Steamer", "Hitters", "Steamer Hitters Dataset.csv"), index=False)

##### Pitchers

In [ ]:
files = [os.path.join(baseball_path, "A03. Steamer", "Pitchers", f"steamer_pitchers_weekly_log_{y}.csv") for y in range(2014, year+1)] + glob.glob(os.path.join(baseball_path, "A03. Steamer", "Pitchers", "2026", "*.csv"))
steamer_pitcher_df = pd.concat([pd.read_csv(f, dtype=str).rename(columns={"1b":"1B","2b":"2B","3b":"3B"}).assign(gamedate=f"{os.path.basename(f)[-12:-8]}-{os.path.basename(f)[-8:-6]}-{os.path.basename(f)[-6:-4]}") if os.path.dirname(f).endswith("2026") else pd.read_csv(f, dtype=str).rename(columns={"1b":"1B","2b":"2B","3b":"3B"}) for f in files], ignore_index=True)
steamer_pitcher_df.to_csv(os.path.join(baseball_path, "A03. Steamer", "Pitchers", "Steamer Pitchers Dataset.csv"), index=False)

##### Cleanup

In [ ]:
del steamer_hitter_df, steamer_pitcher_df
gc.collect()

### A04. Bullpens

Date(s): Today

In [ ]:
historic = False

In [ ]:
_ = Parallel(n_jobs=-1, verbose=0)(delayed(bullpens)(date=date, team_map=team_map, historic=historic) for date in [todaysdate])

### A05. Rosters

##### 1. Batting Orders

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

In [ ]:
_ = Parallel(n_jobs=-1, verbose=0)(delayed(orders)(team_map, df, row) for row in range(len(df)))

##### 2. Rosters

Date(s): Yesterday and Today

In [ ]:
_ = Parallel(n_jobs=-1, verbose=0)(delayed(rosters)(team_map, df, row) for row in range(len(df)))

##### 3. Projected Lineups - RotoGrinders

Date(s): Today

In [ ]:
rotogrinders_lineups_df = scrape_rotogrinders_lineups()
rotogrinders_lineups_df.to_csv(os.path.join(baseball_path, "A05. Rosters", "3. Projected Lineups - RotoGrinders", f"{todaysdate} Projected Lineups - RotoGrinders.csv"), index=False)

 ##### 4. Projected Lineups - Baseball Monster

Date(s): Today

In [ ]:
projected_batting_orders = pd.read_csv(rf"https://baseballmonster.com/Lineups.aspx?csv=1&d={todaysdate_dash}")
projected_batting_orders.to_csv(os.path.join(baseball_path, "A05. Rosters", "4. Projected Lineups - Baseball Monster", f"{todaysdate} Projected Lineups - Baseball Monster.csv"), index=False)

### A06. Weather

##### 1. Open Meteo

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

Setup the Open-Meteo API client **without** cache and with retry on error

In [ ]:
base_session = requests.Session()
retry_session = retry(base_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [ ]:
for date in df['date'].unique():
    print(date)
    if int(date) == int(todaysdate):
        print("Open Meteo: Today")
        open_meteo_df = create_daily_weather_df(openmeteo, df[df['date'] == date])[game_columns + venue_columns + weather_columns + forecast_only_columns]
    else:
        print("Open Meteo: Historic")
        open_meteo_df = create_historic_weather_df(openmeteo, df[df['date'] == date])[game_columns + venue_columns + weather_columns]

    open_meteo_df.to_csv(os.path.join(baseball_path, "A06. Weather", "1. Open Meteo", f"Open Meteo {date}.csv"), index=False)
    time.sleep(2)

##### 2. Kevin

Date(s): Today

In [ ]:
try:
    kevin_df = kevin(todaysdate_dash)
    kevin_df.to_csv(os.path.join(baseball_path, "A06. Weather", "2. Kevin", f"Kevin {todaysdate}.csv"), index=False, encoding='iso-8859-1', errors='ignore')
except:
    print("Could not scrape Kevin weather data.")

### A07. Odds

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

In [ ]:
for date_dash in df['game_date'].unique():
    # Create date compatibility with conventions
    date = date_dash.replace("-", "")
    try:
        # Scrape odds from Sportsbook Review
        odds_df = scrape_sportsbookreview(date_dash)
        odds_df['date'] = date
        # Write to CSV
        odds_df.to_csv(os.path.join(baseball_path, "A07. Odds", "1. Raw", f"Odds {date}.csv"), index=False) 
        # Create DraftKings file
        draftkings_odds_df = select_odds(odds_df, team_dict, sportsbook="draftkings")
        # Write to CSV
        draftkings_odds_df.to_csv(os.path.join(baseball_path, "A07. Odds", "2. DraftKings", f"DraftKings Odds {date}.csv"), index=False)
        # Sleep to avoid hitting rate limit
        time.sleep(5)
    except:
        print(f"Sportsbook Review data unavailable for {date_dash}")

### A08. Projections

Date(s): Yesterday and Today

##### 1. DFF

1. Slates

In [ ]:
dff_slates_df = dff_slates(todaysdate)
dff_slates_df.to_csv(os.path.join(baseball_path, "A08. Projections", "1. DFF", "1. Slates", f"DFF Slates {todaysdate}.csv"), index=False)

In [ ]:
dff_slates_df

2. Projections

In [ ]:
for code in dff_slates_df['URL']:
    try:
        # Scrape projections
        dff_projections_df = dff_projections(todaysdate, code)
        # To CSV
        dff_projections_df.to_csv(os.path.join(baseball_path, "A08. Projections", "1. DFF", "2. Projections", f"DFF Projections {code}.csv"), index=False)
    except KeyError as e:
        print("KeyError")

##### 2. RotoWire

1. Slates

In [ ]:
roto_slates_df = roto_slates(todaysdate)
roto_slates_df.to_csv(os.path.join(baseball_path, "A08. Projections", "2. RotoWire", "1. Slates", f"RotoWire Slates {todaysdate}.csv"), index=False)

2. Projections

In [ ]:
try:
    for code in roto_slates_df['slateID']:
        # Scrape projections
        roto_projections_df = roto_projections(todaysdate, code)
        try:
            # To CSV
            roto_projections_df.to_csv(os.path.join(baseball_path, "A08. Projections", "2. RotoWire", "2. Projections", f"RotoWire Projections {code}.csv"), index=False)
        except:
            print(f"Can't scrape RotoWire projections for slate {code}.")
except:
    print("RotoWire projections unavailable.")

##### Cleanup

In [ ]:
del dff_slates_df, dff_projections_df, roto_slates_df, roto_projections_df
gc.collect()

### A09. DraftKings

Date(s): Yesterday and Today

##### 1. Contests

In [ ]:
# Scrape contests
contest_df = contests(todaysdate)
# Write to CSV
contest_df.to_csv(os.path.join(baseball_path, 'A09. DraftKings', '1. Contests', f'Contests {todaysdate}.csv'), index=False)

##### 7. Subsets

In [ ]:
# # Select subset of contests
subset_df = create_subset(contest_df, contests_per_draftGroupId=5, entry_fee_max=100, added_contestKeys=[], date_dash=todaysdate_dash)
# Write to CSV
subset_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "7. Subsets", f'Subset {todaysdate}.csv'), index=False)

##### 2. Draftables

In [ ]:
# Loop over unique slates in subset of contests
for draftGroupId in list(subset_df['draftGroupId'].unique()):
    try:
        # Scrape draftables (Salaries)
        draftable_df = draftables(draftGroupId)
        # Write to CSV
        draftable_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "2. Draftables", f"Draftables {draftGroupId}.csv"), index=False, encoding='iso-8859-1')
        print(f"Saved {draftGroupId}")
    except:
        print(f"Didn't save {draftGroupId}")

##### 3. Payouts

In [ ]:
# Loop over contests of interest
for i in range(len(subset_df)):
    # Extract contestKey
    contestKey = subset_df['contestKey'][i]
    try:
        # Scrape payouts
        payout_df = payouts(contestKey)
        # Write to CSV
        payout_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "3. Payouts", f"Payouts {contestKey}.csv"), index=False, encoding='iso-8859-1')
    except:
        print(f"Didn't save {contestKey}.")


##### 4. Results, 5. Entry Results, and 6. Player Results

In [ ]:
try:
    yesterdays_subset_df = pd.read_csv(os.path.join(baseball_path, 'A09. DraftKings', '7. Subsets', f'Subset {yesterdaysdate}.csv'))

    # Loop over yesterday's contests
    for i in range(len(yesterdays_subset_df)):
        # Extract contestKey
        contestKey = yesterdays_subset_df['contestKey'][i]
        print(contestKey)
        # 4. Results
        # Download
        results(contestKey, sleep_time=5)

        try:
            # Read into pandas, if possible
            results_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "4. Results", f"contest-standings-{contestKey}.csv"))

            # 5. Entry Results
            # Extract contest results
            entry_results_df = entry_results(results_df)
            # Write to CSV
            entry_results_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "5. Entry Results", f"Entry Results {contestKey}.csv"), index=False, encoding='iso-8859-1')

            # 6. Player Results
            # Extract player results
            player_results_df = player_results(results_df)
            # Write to CSV
            player_results_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "6. Player Results", f"Player Results {contestKey}.csv"), index=False, encoding='iso-8859-1')
        except IndexError as e:
            print(f"Downloaded contest-standings-{contestKey}. Non-traditional format.")
        except pd.errors.EmptyDataError as e:
            print(f"Downloaded contest-standings-{contestKey}. Empty file.")
        except FileNotFoundError as e:
            print(f"Couldn't find {contestKey}.")
except FileNotFoundError as e:
    print(f"Couldn't find yesterday's subset file.")

##### Cleanup

In [ ]:
del contest_df, subset_df, draftable_df, payout_df, results_df, entry_results_df, player_results_df, yesterdays_subset_df
gc.collect()

### M01. Batted Balls

Date(s): All

In [ ]:
%%time
is_main = False

%run "C:\Users\James\Documents\MLB\Code\M01. Batted Balls.ipynb"

### M01. Weather Factors

Date(s): All

In [ ]:
%%time
is_main = False

%run "C:\Users\James\Documents\MLB\Code\M01. Weather Factors.ipynb"

### B01. Matchups

Date(s): Yesterday and Today

In [ ]:
start_date = yesterdaysdate

In [ ]:
df = game_df[(game_df['date'].astype(int) >= int(yesterdaysdate)) & (game_df['date'].astype(int) <= int(todaysdate))].reset_index(drop=True)

In [ ]:
%%time
create_all_matchups(df, baseball_path, team_dict, batter_inputs, pitcher_inputs, start_date, n_jobs=1)

### B03. Contest Guides

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

In [ ]:
for date in df['date'].unique():
    print(f"Generating contest guides for {date}.")
    # Read in contest subset
    subset_df = pd.read_csv(os.path.join(baseball_path, 'A09. DraftKings', '7. Subsets', f'Subset {date}.csv'))

    # Loop over contestKeys
    for contestKey in subset_df['contestKey'].reset_index(drop=True):
        print(contestKey)
        try:
            guide = contest_guide(df, subset_df=subset_df, contestKey=contestKey)
            if not guide.empty:
                guide.to_csv(os.path.join(baseball_path, "B03. Contest Guides", f"Contest Guide {contestKey}.csv"), index=False)
            else:
                print(f"Contest Guide {contestKey} is empty.")
        except FileNotFoundError as e:
            print(f"Draftables {contestKey}.csv not found.")